# Estratégia 4 — EV/EBITDA + Dívida Líquida/EBITDA
**Competição de Carteira de Investimentos 2026 — PPGOLD/UFPR**

---
## Por que EV/EBITDA supera o P/L como métrica de Valuation?

O **P/L (Preço/Lucro)** é a métrica mais popular, mas tem falhas graves como indicador de Valuation:
- Lucro líquido é afetado por **depreciação, amortização e estrutura de capital** — fatores contábeis que variam entre empresas e setores
- Empresas alavancadas parecem 'baratas' por P/L, mas são caras quando consideramos a dívida
- Imposto de renda e benefícios fiscais distorcem a comparação cross-setor

### A solução: Enterprise Value / EBITDA

| Componente | Definição | Por que importa |
|---|---|---|
| **EV** (Enterprise Value) | Capitalização de Mercado + Dívida Líquida | Preço real de aquisição da empresa |
| **EBITDA** | Lucro antes de Juros, Impostos, Dep. e Amort. | Proxy de geração operacional de caixa |
| **EV/EBITDA** | Quantos anos de EBITDA para pagar o EV | Comparável entre setores e estruturas de capital |

**Damodaran (2012):** *'EV/EBITDA é o múltiplo mais consistente para comparação cross-setor porque remove os efeitos da estrutura de capital e das políticas contábeis.'*

### Segunda métrica: Dívida Líquida / EBITDA

Mede o **endividamento relativo à capacidade de geração de caixa**. Em quantos anos de operação a empresa quita toda a sua dívida?
- ≤ 1.5x → Empresa pouco alavancada (muito saudável)
- 1.5x – 3.0x → Alavancagem moderada (aceitável)
- > 4.0x → Empresa excessivamente endividada (risco alto, especialmente com SELIC elevada)


---
## Etapa 1 — Importação de Bibliotecas

In [ ]:
import requests, io, warnings
import pandas as pd
import numpy as np
import yfinance as yf
from itertools import combinations
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
print('Bibliotecas carregadas.')

---
## Etapa 2 — Coleta do Universo B3 e Extração de EV/EBITDA e Dívida

In [ ]:
headers = {'User-Agent': 'Mozilla/5.0'}
df_b3 = pd.read_html(io.StringIO(
    requests.get('https://www.fundamentus.com.br/resultado.php', headers=headers).text
), decimal=',', thousands='.')[0]

def pct(s):
    return pd.to_numeric(s.astype(str).str.replace('.','',regex=False)
        .str.replace(',','.',regex=False).str.replace('%','',regex=False),
        errors='coerce') / 100.0

for col in ['ROE','ROIC','Mrg Ebit','Mrg. Líq.','Div.Yield','Cresc. Rec.5a']:
    df_b3[col] = pct(df_b3[col])
for col in ['P/L','P/VP','EV/EBITDA','EV/EBIT','Liq.2meses','Dív.Brut/ Patrim.','Liq. Corr.']:
    df_b3[col] = pd.to_numeric(df_b3[col], errors='coerce')

print(f'Total de ativos carregados: {len(df_b3)}')
df_b3[['Papel','EV/EBITDA','Dív.Brut/ Patrim.','ROE','Liq.2meses']].head(5)

---
## Etapa 3 — Filtros de Qualidade e Liquidez

Aplicamos filtros estritos que combinam os critérios:
1. **Liquidez** > R$ 5 milhões bimestral (cumprimento do edital)
2. **EV/EBITDA** entre 3x e 15x — empresas nem destruídas nem caríssimas
3. **Dívida/PL** < 2.0x — controle de alavancagem (resistência a SELIC alta)
4. **ROE** > 10% — retorno mínimo sobre patrimônio
5. **Margem EBIT** positiva — lucratividade operacional confirmada


In [ ]:
df = df_b3[
    (df_b3['Liq.2meses'] > 5_000_000)   &  # Alta liquidez
    (df_b3['EV/EBITDA'] > 2)            &  # EV/EBITDA faz sentido (positivo)
    (df_b3['EV/EBITDA'] < 20)           &  # Sem empresas absurdamente caras
    (df_b3['Dív.Brut/ Patrim.'] < 2.0)  &  # Dívida controlada vs PL
    (df_b3['ROE'] > 0.10)               &  # ROE > 10%
    (df_b3['Mrg Ebit'] > 0)                 # Lucro operacional positivo
].copy().reset_index(drop=True)

print(f'Ativos após filtros de qualidade: {len(df)}')

---
## Etapa 4 — Score Composto: EV/EBITDA × Dívida × ROE

Construímos um **Score Composto** que penaliza empresas caras (EV/EBITDA alto), endividadas (Dívida/PL alto) e premia as mais rentáveis (ROE alto):

```
Score Final = rank(EV/EBITDA) + rank(Dív/PL) – 2 × rank(ROE)
```
*(Quanto menor o Score, mais atrativo o ativo)*


In [ ]:
df['rank_ev_ebitda']   = df['EV/EBITDA'].rank(ascending=True)         # Menor = mais barato
df['rank_divida']      = df['Dív.Brut/ Patrim.'].rank(ascending=True)  # Menor = menos endividado
df['rank_roe']         = df['ROE'].rank(ascending=False)               # Maior ROE = melhor
df['rank_margem']      = df['Mrg Ebit'].rank(ascending=False)          # Maior margem = melhor

# Score composto (menor = melhor)
df['Score_Composto'] = (
    df['rank_ev_ebitda'] * 0.35 +   # EV/EBITDA tem peso maior (valuation)
    df['rank_divida']   * 0.25 +   # Endividamento
    df['rank_roe']      * 0.25 +   # Rentabilidade
    df['rank_margem']   * 0.15     # Eficiência operacional
)

df_rank = df.sort_values('Score_Composto').head(15)
print('Top 15 por Score Composto EV/EBITDA:')
display(df_rank[['Papel','EV/EBITDA','Dív.Brut/ Patrim.','ROE','Mrg Ebit','Score_Composto']]
        .reset_index(drop=True))

---
## Etapa 5 — Visualização: Quadrante Valuation × Endividamento

O gráfico abaixo posiciona as empresas em um quadrante onde o **canto superior esquerdo é o ideal**: menor EV/EBITDA (mais barato) e menor Dívida (mais saudável). O tamanho do ponto representa o ROE.


In [ ]:
top_plot = df.nsmallest(30, 'Score_Composto').copy()

plt.figure(figsize=(12, 7))
scatter = plt.scatter(
    top_plot['EV/EBITDA'],
    top_plot['Dív.Brut/ Patrim.'],
    s=top_plot['ROE'] * 1500 + 50,
    c=top_plot['Score_Composto'], cmap='RdYlGn_r', alpha=0.7, edgecolors='gray'
)
plt.colorbar(scatter, label='Score Composto (menor = melhor)')

for _, row in top_plot.iterrows():
    plt.annotate(row['Papel'], (row['EV/EBITDA'], row['Dív.Brut/ Patrim.']),
                 fontsize=7, ha='center', va='bottom')

plt.xlabel('EV/EBITDA (Valuation — menor é melhor)')
plt.ylabel('Dívida Bruta / PL (Endividamento)')
plt.title('Quadrante de Valuation × Endividamento\n(Tamanho = ROE)')
plt.tight_layout()
plt.show()

---
## Etapa 6 — Top 12 Candidatos e Otimização Combinatória

In [ ]:
top12 = df.nsmallest(12, 'Score_Composto')
tickers_top = [t + '.SA' for t in top12['Papel'].tolist()]
rf = 0.1075

precos_5a = yf.download(tickers_top, start='2021-03-26', end='2026-03-26', progress=False)['Close']
precos_5a = precos_5a.dropna(axis=1, thresh=int(len(precos_5a)*0.8)).ffill().dropna()
tickers_v = precos_5a.columns.tolist()

ret_d = precos_5a.pct_change().dropna()
mu = ret_d.mean() * 252
cov = ret_d.cov() * 252

def neg_sharpe(w, mu, cov, rf):
    r = np.dot(w, mu); v = np.sqrt(w @ cov @ w)
    return -(r - rf)/v if v > 0 else 0

best_s, best_c, best_w = -np.inf, None, None
for comb in combinations(tickers_v, 4):
    mu_c = mu[list(comb)].values
    cov_c = cov.loc[list(comb), list(comb)].values
    res = minimize(neg_sharpe, [0.25]*4, args=(mu_c, cov_c, rf),
                   method='SLSQP', bounds=[(0,1)]*4,
                   constraints={'type':'eq','fun':lambda w: w.sum()-1})
    if -res.fun > best_s:
        best_s = -res.fun; best_c = list(comb); best_w = res.x

mu_f = mu[best_c].values
cov_f = cov.loc[best_c, best_c].values
ret_f, vol_f = np.dot(best_w, mu_f), np.sqrt(best_w @ cov_f @ best_w)

print('='*55)
print('CARTEIRA ÓTIMA — EV/EBITDA + DÍVIDA + MARKOWITZ')
print('='*55)
for t, w in zip(best_c, best_w):
    papel = t.replace('.SA','')
    ev = df[df['Papel']==papel]['EV/EBITDA'].values
    ev_str = f'(EV/EBITDA={ev[0]:.1f}x)' if len(ev) > 0 else ''
    print(f'  {papel}: {w*100:.2f}% {ev_str}')
print(f'\n  Retorno Esperado : {ret_f*100:.2f}% a.a.')
print(f'  Volatilidade     : {vol_f*100:.2f}% a.a.')
print(f'  Índice de Sharpe : {best_s:.4f}')

---
## Etapa 7 — Fronteira Eficiente

In [ ]:
n_sim = 10000
rets, vols, sharpes = [], [], []
for _ in range(n_sim):
    w = np.random.dirichlet(np.ones(4))
    r, v = np.dot(w, mu_f), np.sqrt(w @ cov_f @ w)
    rets.append(r); vols.append(v); sharpes.append((r-rf)/v)

plt.figure(figsize=(10, 6))
sc = plt.scatter(vols, rets, c=sharpes, cmap='coolwarm', s=8, alpha=0.4)
plt.colorbar(sc, label='Índice de Sharpe')
plt.scatter(vol_f, ret_f, marker='*', color='red', s=300, zorder=5,
            label=f'Portfólio Ótimo (Sharpe={best_s:.2f})')
plt.axhline(rf, color='gray', linestyle='--', label=f'CDI ({rf*100:.1f}%)')
plt.xlabel('Risco Anualizado'); plt.ylabel('Retorno Anualizado')
plt.title('Fronteira Eficiente — Estratégia EV/EBITDA + Dívida')
plt.legend(); plt.tight_layout(); plt.show()

---
## Etapa 8 — Indicadores Exigidos pelo Edital

In [ ]:
papeis_f = [t.replace('.SA','') for t in best_c]
df_ind = df[df['Papel'].isin(papeis_f)][['Papel','ROE','EV/EBITDA','P/VP','Dív.Brut/ Patrim.']].copy()
df_ind.columns = ['Ativo','ROE (Rentabilidade)','EV/EBITDA (Valuation)','P/VP (Valuation)','Dív/PL (Endividamento)']
df_ind['ROE (Rentabilidade)'] = df_ind['ROE (Rentabilidade)'].map('{:.1%}'.format)
display(df_ind.reset_index(drop=True))

---
## Conclusão da Estratégia EV/EBITDA

A métrica EV/EBITDA, endossada por Damodaran (referência central da bibliografia do plano de ensino), oferece uma perspectiva de Valuation muito superior ao P/L clássico precisamente por incluir a **estrutura de capital** na avaliação do preço. Ao combinar EV/EBITDA com Dívida Líquida/EBITDA e ROE, criamos um Score Composto que identifica empresas simultaneamente **baratas, saudáveis e rentáveis** — o trio ideal para um portfólio competitivo frente ao CDI elevado do contexto macroeconômico brasileiro em 2026.
